In [1]:
#Futures-Based Real TTF NG Price Forecasts
#Formula used:
#  R_hat(t+h) = R_t * (f_t^h / s_t) / (1 + pi_bar_t)^h
#Log approximations are used, which transforms this formula into an equivalent formula which uses logs.

import pandas as pd
import numpy as np
from dateutil.relativedelta import relativedelta

In [2]:
# ── Load input panel ───────────────────────────────────────────────────────────
 
df = pd.read_excel("Input_for_futures_based_model.xlsx",
                   parse_dates=["forecast_origin"])
 
print(f"Loaded {len(df)} rows  |  "
      f"{df['forecast_origin'].nunique()} forecast origins  |  "
      f"{df['horizon'].nunique()} horizons")

Loaded 1512 rows  |  168 forecast origins  |  9 horizons


In [3]:
# ── Apply formula ───────────────────────────────────────────
#
# Step 1: futures basis  = f_t^h / s_t
# Step 2: inflation deflator = h × π̄_t
# Step 3: forecast = R_t * (basis - deflator)
df["futures_basis"]  = df["futures_price"] / df["nominal_price"]
df["inflation_term"] = df["horizon"] * df["avg_inflation"]
df["forecast"]       = df["real_price"] * (df["futures_basis"] - df["inflation_term"])

In [4]:
# ── Compute target month (year-month of the actual realisation) ────────────────
# actual_month = forecast_origin month + h months
# It is stored as a YYYY-MM string for matching — avoids business vs calendar day mismatches entirely
 
df["actual_month"] = df.apply(
    lambda r: (r["forecast_origin"] + relativedelta(months=int(r["horizon"]))).strftime("%Y-%m"),
    axis=1
)
 
# ── Build actuals lookup keyed on year-month ───────────────────────────────────
# Each forecast_origin represents the last business day of that month,
#so its real_price is the actual for that calendar month.
#The price used is monthly average real TTF NG price as for all ther other models.
 
actuals_lookup = (
    df[["forecast_origin", "real_price"]]
    .drop_duplicates("forecast_origin")
    .assign(actual_month=lambda x: x["forecast_origin"].dt.strftime("%Y-%m"))
    .rename(columns={"real_price": "actual"})
    [["actual_month", "actual"]]
)
 
# ── Merge actuals on year-month key ───────────────────────────────────────────
 
df = df.merge(actuals_lookup, on="actual_month", how="left")

In [5]:
# ── Select output columns ─────────────────────────────────────────────────────
 
out = df[["forecast_origin", "horizon", "actual_month", "forecast", "actual"]].copy()
out.insert(2, "model", "futures")
 
# ── Diagnostics ───────────────────────────────────────────────────────────────
 
total     = len(out)
valid_f   = out["forecast"].notna().sum()
valid_a   = out["actual"].notna().sum()
missing_f = out["forecast"].isna().sum()
 
print(f"\nTotal rows:              {total}")
print(f"Valid forecasts:         {valid_f} ({100*valid_f/total:.1f}%)")
print(f"Missing forecasts:       {missing_f} ({100*missing_f/total:.1f}%)")
print(f"Actuals available:       {valid_a} ({100*valid_a/total:.1f}%)")
print(f"Actuals missing (future):{total - valid_a} ({100*(total-valid_a)/total:.1f}%)")
 
print("\nCoverage by horizon (% with valid forecast):")
coverage = out.groupby("horizon")["forecast"].apply(
    lambda s: f"{100*s.notna().mean():.1f}%  ({s.notna().sum()} obs)"
)
print(coverage.to_string())
 
print("\nFirst 18 rows:")
print(out.head(18).to_string(index=False))


Total rows:              1512
Valid forecasts:         1430 (94.6%)
Missing forecasts:       82 (5.4%)
Actuals available:       1403 (92.8%)
Actuals missing (future):109 (7.2%)

Coverage by horizon (% with valid forecast):
horizon
1     92.9%  (156 obs)
3     93.5%  (157 obs)
6     94.6%  (159 obs)
9     95.2%  (160 obs)
12    96.4%  (162 obs)
15    95.8%  (161 obs)
18    95.2%  (160 obs)
21    94.0%  (158 obs)
24    93.5%  (157 obs)

First 18 rows:
forecast_origin  horizon   model actual_month  forecast    actual
     2012-01-31        1 futures      2012-02       NaN 27.188250
     2012-01-31        3 futures      2012-04       NaN 25.337329
     2012-01-31        6 futures      2012-07 22.485987 24.737803
     2012-01-31        9 futures      2012-10       NaN 27.000380
     2012-01-31       12 futures      2013-01 27.707047 26.723578
     2012-01-31       15 futures      2013-04       NaN 28.288246
     2012-01-31       18 futures      2013-07 24.275670 26.091651
     2012-01-31  

In [6]:
# ── Save ──────────────────────────────────────────────────────────────────────
 
#out.to_csv("futures_forecasts_long.csv", index=False)
out.to_excel("futures_forecasts_long.xlsx", index=False)
 
print("\nSaved:")
#print("  futures_forecasts_long.csv")
print("  futures_forecasts_long.xlsx")


Saved:
  futures_forecasts_long.xlsx
